# 02 · Hallucinations & Grounding (RAG)
### *Ethics, Safety & the Future of LLMs — Unit 5*

An LLM optimizes for the **most plausible** next token, not the **true** one — so it will confidently invent facts.
Here we watch a model hallucinate, then **ground** it with retrieval so it answers from real text.

1. Ungrounded answer to a tricky question.
2. Retrieve a relevant document and answer **from context** (RAG).
3. A simple **faithfulness** check.

> CPU is fine.

In [ ]:
!pip -q install "transformers>=4.40" sentence-transformers sentencepiece

In [ ]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util

gen = pipeline("text2text-generation", model="google/flan-t5-base")
enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def ask(prompt, n=64):
    return gen(prompt, max_new_tokens=n)[0]["generated_text"]

## 1 · Ungrounded — the model guesses

With no source to consult, the model produces a fluent answer that may be **wrong or fabricated**, especially for
obscure or false-premise questions.

In [ ]:
q = "In what year did the fictional company Zenthara Corp release its first quantum phone?"
print("UNGROUNDED:", ask(f"Answer the question. {q}"))

## 2 · Grounded (RAG) — the model reads first

We give the model a small document store, **retrieve** the most relevant passage with embeddings, and instruct it
to answer **only from that context** — and to say it doesn't know otherwise.

In [ ]:
docs = [
    "The Eiffel Tower was completed in 1889 and stands 330 metres tall in Paris.",
    "Photosynthesis converts sunlight, water and CO2 into glucose and oxygen in plant chloroplasts.",
    "The GDP of Japan in 2022 was approximately 4.2 trillion US dollars.",
    "Mount Everest, at 8,849 metres, is the highest mountain above sea level.",
]

def rag_answer(question):
    d_emb = enc.encode(docs, convert_to_tensor=True)
    q_emb = enc.encode(question, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, d_emb)[0]
    top = int(scores.argmax()); ctx = docs[top]
    prompt = (f"Context: {ctx}\n"
              f"Answer the question using ONLY the context. "
              f"If the context does not contain the answer, reply 'I don't know'.\n"
              f"Question: {question}\nAnswer:")
    return ctx, ask(prompt)

for question in ["How tall is the Eiffel Tower?",
                 "What year did Zenthara Corp release a quantum phone?"]:
    ctx, ans = rag_answer(question)
    print("Q:", question)
    print("  retrieved:", ctx[:60], "...")
    print("  answer   :", ans, "\n")

Notice the grounded model **refuses** the unanswerable question instead of inventing a fact — grounding plus an
explicit "say I don't know" instruction is the single biggest hallucination lever.

## 3 · Faithfulness check

A cheap guardrail: verify that the answer's content actually appears in the retrieved context.

In [ ]:
def faithful(answer, context):
    a = set(w.lower().strip(".,") for w in answer.split() if len(w) > 3)
    c = set(w.lower().strip(".,") for w in context.split())
    overlap = len(a & c) / max(1, len(a))
    return overlap

ctx, ans = rag_answer("How tall is the Eiffel Tower?")
print(f"answer: {ans}")
print(f"faithfulness (content overlap with source): {faithful(ans, ctx):.2f}")

## Recap & your turn

- Hallucination is a side-effect of the training objective, not a bug you can fully patch.
- **RAG** (retrieve → answer from context) plus an explicit *"say I don't know"* dramatically reduces it.
- Always keep a **human in the loop** for high-stakes answers.

**Exercises**
1. Add more documents and questions; where does retrieval pick the wrong passage?
2. Return the **citation** (which doc) alongside the answer.
3. Explore `truthful_qa` from `datasets` — a benchmark built around common human misconceptions.